# Hierarchical Indices for RAG — From First Principles

Flat vector indices treat every chunk as if it lived at the same semantic altitude. That is convenient, but documents are not flat. Reports, books, manuals, and policies are naturally hierarchical: document → chapter → section → paragraph.

Hierarchical retrieval preserves that structure. Instead of jumping directly from query to paragraph, we can first identify the relevant region of the corpus, then zoom in. This notebook shows how to build that pipeline from scratch with summaries, section routing, and multi-level FAISS indices.

## Why Flat Retrieval Loses Document Structure

Suppose a document has one section on transmission, one on storage, and one on policy. A flat index lets any paragraph compete with any other paragraph. That can work, but it ignores two useful priors:

1. some sections are globally more relevant than others,
2. once a section is selected, local detail retrieval should happen inside that section.

Hierarchical indices encode those priors explicitly.

## The Two-Level Pattern

```
query
  -> summary index (coarse routing)
  -> chosen sections
  -> detail index inside those sections
  -> grounded answer
```

This architecture works because summaries are cheap routing documents. They compress the semantic role of a section while preserving enough meaning to route the query into the right neighborhood.

## Information Retrieval Theory Behind Hierarchical Search

Hierarchical indices are not a new idea invented for RAG. They mirror decades of
research in databases and information retrieval.

### Complexity argument

A flat scan over **N** chunks costs **O(N)** per query. A two-level hierarchy with
**G** groups of roughly **N/G** chunks each costs **O(G + N/G)**, which is minimised
when **G ≈ √N**, giving **O(√N)**. A balanced **k-ary tree** of depth **d** costs
**O(k · d)** where **d = log_k N**, so retrieval becomes **O(k · log_k N)** — the
same logarithmic scaling that B-trees and skip lists exploit in databases.

### Analogy to database indexing

| Concept | Database world | RAG world |
|---|---|---|
| Leaf nodes | Data pages | Detail chunks |
| Internal nodes | Index pages / B-tree nodes | Summary embeddings |
| Traversal | Key comparisons down the tree | Cosine similarity at each level |
| Branching factor | Page fan-out (100–1000) | Top-k at each level (2–10) |

### Complexity comparison table

| Strategy | Comparisons per query | When it wins |
|---|---|---|
| Flat scan | O(N) | Tiny corpora (N < 50) |
| Two-level | O(√N) | Medium corpora with clear sections |
| k-level tree | O(k · log_k N) | Large corpora with deep structure |
| Approximate NN (HNSW) | O(log N) | Huge corpora, no document structure |

The key insight is that hierarchical indices **combine structural pruning with**
**semantic matching**, whereas ANN methods like HNSW use only geometric proximity.

In [ ]:
import math

def complexity_comparison(n_chunks, group_size=None, k_levels=2, branching=3):
    """Compare search costs for flat vs hierarchical strategies."""
    flat = n_chunks
    g = group_size or int(math.isqrt(n_chunks))
    two_level = g + (n_chunks // g)
    k_tree = branching * int(math.log(n_chunks, branching)) if n_chunks > 1 else 1
    return {
        "flat_O_N": flat,
        "two_level_O_sqrtN": two_level,
        "tree_O_k_logk_N": k_tree,
    }

print(f"{'N':>8} | {'Flat':>8} | {'2-Level':>8} | {'Tree':>8}")
print('-' * 42)
for n in [20, 100, 500, 2000, 10000]:
    c = complexity_comparison(n)
    print(f"{n:>8} | {c['flat_O_N']:>8} | {c['two_level_O_sqrtN']:>8} | {c['tree_O_k_logk_N']:>8}")

## Environment Setup

All notebooks in this overhaul use the same minimal native stack:

- **Qwen/Qwen3-14B** for generation
- **BAAI/bge-base-en-v1.5** for embeddings
- **FAISS** for similarity search
- **Google Drive** for persistent Colab caching

The goal is to keep the pipeline transparent. Every major component is visible as plain Python functions instead of framework abstractions.

In [ ]:
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy
print("Installed core native RAG dependencies.")

In [ ]:
import time
import math
import json
import re
from collections import defaultdict, Counter

import numpy as np
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount("/content/drive")
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto"
)
print("Loaded", MODEL_NAME)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR)
print("Loaded embedding model:", embed_model.__class__.__name__)

In [ ]:
def generate(prompt, max_new_tokens=220, temperature=0.2):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        top_k=20,
        temperature=temperature,
        do_sample=temperature > 0,
    )
    answer_ids = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(answer_ids, skip_special_tokens=True).strip()

print("Generation helper ready.")

## A Structured Corpus with Sections and Subsections

We will use a synthetic energy-systems handbook with explicit section names. That makes the hierarchy visible instead of implicit.

In [ ]:
corpus = {
    "Grid Planning": {
        "Transmission": [
            "Transmission expansion connects distant renewable resources and reduces congestion costs.",
            "High-voltage corridors can lower local storage needs by sharing balancing resources across regions.",
        ],
        "Distribution": [
            "Distribution upgrades are critical when rooftop solar and electric vehicle loads concentrate on the same feeders.",
            "Flexible inverters can reduce local voltage problems before expensive hardware upgrades are required.",
        ],
    },
    "Flexibility Resources": {
        "Batteries": [
            "Batteries provide millisecond response and excel at frequency regulation and short-duration shifting.",
            "Battery economics depend on cycle frequency, arbitrage spreads, and ancillary service markets.",
        ],
        "Demand Response": [
            "Demand response shifts or sheds load during system peaks using industrial, commercial, and residential flexibility.",
            "It works especially well when peak prices are sharp and flexible loads can be automated.",
        ],
    },
    "Policy and Markets": {
        "Capacity Design": [
            "Capacity markets reward dependable availability during stress periods rather than annual energy output alone.",
            "Poorly designed capacity rules can undervalue demand response and storage compared with thermal units.",
        ],
        "Performance Incentives": [
            "Performance payments reward resources that deliver during net-load ramps and evening peaks.",
            "These incentives often favor fast flexibility and accurate forecasting rather than nameplate capacity.",
        ],
    },
}

for chapter, sections in corpus.items():
    print("\nCHAPTER:", chapter)
    for section, paragraphs in sections.items():
        print("  -", section, "| paragraphs:", len(paragraphs))

## Building Chunks and Section Summaries

Each paragraph becomes a detail chunk. Each section also gets a summary record that represents the whole section. In production, those summaries can be generated by an LLM. Here we use the LLM directly so the routing index is derived from the underlying text rather than hand-written.

In [ ]:
detail_chunks = []
section_records = []

for chapter, sections in corpus.items():
    for section, paragraphs in sections.items():
        combined = " ".join(paragraphs)
        summary_prompt = f"Summarize this section in 2 concise sentences.\n\nSection: {section}\n\nText: {combined}"
        summary = generate(summary_prompt, max_new_tokens=90)
        section_id = f"{chapter}::{section}"
        section_records.append({
            "section_id": section_id,
            "chapter": chapter,
            "section": section,
            "summary": summary,
        })
        for i, paragraph in enumerate(paragraphs):
            detail_chunks.append({
                "chunk_id": f"{section_id}::p{i}",
                "chapter": chapter,
                "section": section,
                "text": paragraph,
            })

print("Section summaries:", len(section_records))
print("Detail chunks:", len(detail_chunks))
print("Example summary:", section_records[0]["summary"][:180])

In [ ]:
summary_texts = [f"{row['chapter']} | {row['section']} | {row['summary']}" for row in section_records]
summary_embeddings = embed_model.encode(summary_texts, normalize_embeddings=True)
summary_embeddings = np.asarray(summary_embeddings, dtype="float32")
summary_index = faiss.IndexFlatIP(summary_embeddings.shape[1])
summary_index.add(summary_embeddings)

detail_texts = [f"{row['chapter']} | {row['section']} | {row['text']}" for row in detail_chunks]
detail_embeddings = embed_model.encode(detail_texts, normalize_embeddings=True)
detail_embeddings = np.asarray(detail_embeddings, dtype="float32")
detail_index = faiss.IndexFlatIP(detail_embeddings.shape[1])
detail_index.add(detail_embeddings)

print("Summary index size:", summary_index.ntotal)
print("Detail index size:", detail_index.ntotal)

## A Flat Baseline

Before using the hierarchy, we need a flat baseline. This baseline directly searches detail chunks with no summary routing.

In [ ]:
def flat_retrieve(question, k=4):
    q = embed_model.encode([question], normalize_embeddings=True)
    q = np.asarray(q, dtype="float32")
    scores, indices = detail_index.search(q, k)
    return [
        {
            "score": float(score),
            "chunk_id": detail_chunks[idx]["chunk_id"],
            "section": detail_chunks[idx]["section"],
            "text": detail_chunks[idx]["text"],
        }
        for score, idx in zip(scores[0], indices[0])
    ]

for item in flat_retrieve("How can transmission reduce the need for storage?"):
    print(item["section"], round(item["score"], 3), "|", item["text"])

## Two-Level Retrieval

Now we route the query through the summary index first. The retrieved section IDs then constrain the second-stage detail search. This is the core hierarchical pattern.

In [ ]:
def hierarchical_retrieve(question, section_k=2, chunk_k=4):
    q = embed_model.encode([question], normalize_embeddings=True)
    q = np.asarray(q, dtype="float32")

    section_scores, section_indices = summary_index.search(q, section_k)
    selected_sections = {section_records[idx]["section_id"] for idx in section_indices[0]}

    candidates = [row for row in detail_chunks if f"{row['chapter']}::{row['section']}" in selected_sections]
    candidate_texts = [f"{row['chapter']} | {row['section']} | {row['text']}" for row in candidates]
    candidate_embeddings = embed_model.encode(candidate_texts, normalize_embeddings=True)
    candidate_embeddings = np.asarray(candidate_embeddings, dtype="float32")
    local_index = faiss.IndexFlatIP(candidate_embeddings.shape[1])
    local_index.add(candidate_embeddings)

    chunk_scores, chunk_indices = local_index.search(q, min(chunk_k, len(candidates)))
    retrieved = []
    for score, idx in zip(chunk_scores[0], chunk_indices[0]):
        row = candidates[int(idx)]
        retrieved.append({
            "score": float(score),
            "section_id": f"{row['chapter']}::{row['section']}",
            "chunk_id": row["chunk_id"],
            "text": row["text"],
        })
    return selected_sections, retrieved

sections, retrieved = hierarchical_retrieve("How can transmission reduce the need for storage?")
print("Selected sections:", sections)
for item in retrieved:
    print(item["chunk_id"], round(item["score"], 3))

## Retrieval Trace Walkthrough

To understand exactly what happens at each level, let us trace a single query through
the two-level pipeline, printing intermediate similarity scores at every stage.
This makes the routing decisions fully transparent.

In [ ]:
def traced_hierarchical_retrieve(question, section_k=2, chunk_k=4):
    """Two-level retrieval with full score tracing at each stage."""
    q = embed_model.encode([question], normalize_embeddings=True)
    q = np.asarray(q, dtype="float32")

    # Level 1: summary index — coarse routing
    section_scores, section_indices = summary_index.search(q, len(section_records))
    print("=== Level 1: Summary Index (all sections ranked) ===")
    for rank, (score, idx) in enumerate(zip(section_scores[0], section_indices[0])):
        rec = section_records[int(idx)]
        marker = " <-- SELECTED" if rank < section_k else ""
        print(f"  rank {rank+1}: {rec['chapter']}::{rec['section']}  "
              f"score={score:.4f}{marker}")

    # Keep only top-k sections
    selected_sections = {section_records[idx]['section_id'] for idx in section_indices[0][:section_k]}
    print(f"\n==> Routed to sections: {selected_sections}")

    # Level 2: detail chunks within selected sections
    candidates = [row for row in detail_chunks if f"{row['chapter']}::{row['section']}" in selected_sections]
    candidate_texts = [f"{row['chapter']} | {row['section']} | {row['text']}" for row in candidates]
    candidate_embeddings = embed_model.encode(candidate_texts, normalize_embeddings=True)
    candidate_embeddings = np.asarray(candidate_embeddings, dtype="float32")
    local_index = faiss.IndexFlatIP(candidate_embeddings.shape[1])
    local_index.add(candidate_embeddings)
    chunk_scores, chunk_indices = local_index.search(q, min(chunk_k, len(candidates)))

    print(f"\n=== Level 2: Chunk Ranking within {selected_sections} ===")
    for rank, (score, idx) in enumerate(zip(chunk_scores[0], chunk_indices[0])):
        row = candidates[int(idx)]
        print(f"  rank {rank+1}: [{row['chunk_id']}] score={score:.4f}")
        print(f"           {row['text'][:90]}...")

    return selected_sections, [
        {
            "score": float(score),
            "section_id": f"{candidates[int(idx)]['chapter']}::{candidates[int(idx)]['section']}",
            "chunk_id": candidates[int(idx)]["chunk_id"],
            "text": candidates[int(idx)]["text"],
        }
        for score, idx in zip(chunk_scores[0], chunk_indices[0])
    ]

trace_query = "How can transmission reduce the need for storage?"
print(f'Query: "{trace_query}"\n')
traced_sections, traced_chunks = traced_hierarchical_retrieve(trace_query)

## Adding a Third Level

A two-level hierarchy is often enough, but some corpora benefit from another layer. Here we treat chapters as a level above sections. The query first routes to the most relevant chapter, then to sections, then to detail chunks.

In [ ]:
chapter_records = []
for chapter, sections in corpus.items():
    text = " ".join(" ".join(paragraphs) for paragraphs in sections.values())
    chapter_summary = generate(f"Summarize this chapter in 2 concise sentences.\n\n{text}", max_new_tokens=90)
    chapter_records.append({"chapter": chapter, "summary": chapter_summary})

chapter_embeddings = embed_model.encode(
    [f"{row['chapter']} | {row['summary']}" for row in chapter_records],
    normalize_embeddings=True,
)
chapter_embeddings = np.asarray(chapter_embeddings, dtype="float32")
chapter_index = faiss.IndexFlatIP(chapter_embeddings.shape[1])
chapter_index.add(chapter_embeddings)

for row in chapter_records:
    print(row["chapter"], "->", row["summary"][:100])

In [ ]:
def three_level_retrieve(question, chapter_k=1, section_k=2, chunk_k=4):
    q = embed_model.encode([question], normalize_embeddings=True)
    q = np.asarray(q, dtype="float32")

    chapter_scores, chapter_indices = chapter_index.search(q, chapter_k)
    selected_chapters = {chapter_records[idx]["chapter"] for idx in chapter_indices[0]}

    valid_sections = [row for row in section_records if row["chapter"] in selected_chapters]
    section_texts = [f"{row['chapter']} | {row['section']} | {row['summary']}" for row in valid_sections]
    section_embeddings = embed_model.encode(section_texts, normalize_embeddings=True)
    section_embeddings = np.asarray(section_embeddings, dtype="float32")
    local_section_index = faiss.IndexFlatIP(section_embeddings.shape[1])
    local_section_index.add(section_embeddings)

    section_scores, section_indices = local_section_index.search(q, min(section_k, len(valid_sections)))
    selected_sections = {f"{valid_sections[idx]['chapter']}::{valid_sections[idx]['section']}" for idx in section_indices[0]}

    candidates = [row for row in detail_chunks if f"{row['chapter']}::{row['section']}" in selected_sections]
    candidate_texts = [f"{row['chapter']} | {row['section']} | {row['text']}" for row in candidates]
    candidate_embeddings = embed_model.encode(candidate_texts, normalize_embeddings=True)
    candidate_embeddings = np.asarray(candidate_embeddings, dtype="float32")
    local_chunk_index = faiss.IndexFlatIP(candidate_embeddings.shape[1])
    local_chunk_index.add(candidate_embeddings)
    chunk_scores, chunk_indices = local_chunk_index.search(q, min(chunk_k, len(candidates)))

    return {
        "chapters": selected_chapters,
        "sections": selected_sections,
        "chunks": [candidates[idx] for idx in chunk_indices[0]],
    }

trace = three_level_retrieve("Why might performance payments favor batteries and demand response?")
print("Chapters:", trace["chapters"])
print("Sections:", trace["sections"])
for chunk in trace["chunks"]:
    print(chunk["chunk_id"])

## Routing Traces Make the Hierarchy Inspectable

A good hierarchical retriever should not be a mystery. We should be able to see the chapter choice, the section choice, and the final chunk choice. That is one of the biggest benefits of explicit hierarchy over hidden chain components.

In [ ]:
test_queries = [
    "How can transmission reduce the need for storage?",
    "Why do performance payments often reward fast flexibility?",
    "When is demand response especially useful?",
    "What are the tradeoffs of battery economics?",
]

for q in test_queries:
    trace = three_level_retrieve(q)
    print("\nQUESTION:", q)
    print("CHAPTERS:", trace["chapters"])
    print("SECTIONS:", trace["sections"])
    print("CHUNK IDS:", [c["chunk_id"] for c in trace["chunks"]])

## Flat vs Hierarchical Comparison

The point of hierarchy is not to make retrieval more ceremonial. The point is to push the detail search into a smaller, semantically cleaner neighborhood. The next cell compares the result sets.

In [ ]:
comparison_query = "How do policy incentives reward fast flexibility during evening peaks?"

print("FLAT")
for item in flat_retrieve(comparison_query):
    print(item["section"], round(item["score"], 3))

print("\nHIERARCHICAL")
_, hier_items = hierarchical_retrieve(comparison_query)
for item in hier_items:
    print(item["section_id"], round(item["score"], 3))

## Edge Cases and Failure Modes

Hierarchical retrieval is not universally better. Understanding when it fails is just
as important as knowing when it helps.

- **Cross-section answers**: When the correct answer spans two sections, the router
  may pick only one, missing half the evidence.
- **Ambiguous routing**: When the query is broad enough to match many summaries equally,
  the coarse level adds noise rather than focus.
- **Hierarchy helps dramatically**: When the query targets a specific topic clearly
  nested inside one section, pruning irrelevant sections boosts precision.

In [ ]:
# --- Case 1: Hierarchy helps — query clearly targets one section ---
q_focused = "What millisecond response do batteries provide for frequency regulation?"
print('Case 1: FOCUSED QUERY (hierarchy should help)')
print(f'Query: "{q_focused}"')
flat_results = flat_retrieve(q_focused, k=4)
_, hier_results = hierarchical_retrieve(q_focused, section_k=2, chunk_k=4)
flat_sections = [r['section'] for r in flat_results]
hier_sections = [r['section_id'] for r in hier_results]
print(f'  Flat sections hit:  {flat_sections}')
print(f'  Hier sections hit:  {hier_sections}')
flat_top = flat_results[0]['score'] if flat_results else 0
hier_top = hier_results[0]['score'] if hier_results else 0
print(f'  Flat top score:  {flat_top:.4f}')
print(f'  Hier top score:  {hier_top:.4f}')

# --- Case 2: Cross-section answer — hierarchy may miss evidence ---
q_cross = "How do transmission upgrades and demand response work together to handle peaks?"
print(f'\nCase 2: CROSS-SECTION QUERY (hierarchy may lose evidence)')
print(f'Query: "{q_cross}"')
flat_results = flat_retrieve(q_cross, k=4)
_, hier_results = hierarchical_retrieve(q_cross, section_k=1, chunk_k=4)
flat_sections = set(r['section'] for r in flat_results)
hier_sections = set(r['section_id'] for r in hier_results)
print(f'  Flat sections hit:  {flat_sections}')
print(f'  Hier sections hit:  {hier_sections}')
print(f'  Flat covers {len(flat_sections)} sections vs Hier covers {len(hier_sections)} section(s)')
if len(flat_sections) > len(hier_sections):
    print('  ⚠ Hierarchy missed sections — cross-section failure mode')

# --- Case 3: Ambiguous routing — broad query matches many summaries ---
q_ambig = "What are the main challenges in modern energy systems?"
print(f'\nCase 3: AMBIGUOUS QUERY (many summaries match equally)')
print(f'Query: "{q_ambig}"')
q_vec = embed_model.encode([q_ambig], normalize_embeddings=True)
q_vec = np.asarray(q_vec, dtype='float32')
all_scores, all_indices = summary_index.search(q_vec, len(section_records))
print('  Summary scores (all sections):')
for score, idx in zip(all_scores[0], all_indices[0]):
    rec = section_records[int(idx)]
    print(f"    {rec['chapter']}::{rec['section']}  score={score:.4f}")
score_range = float(all_scores[0][0]) - float(all_scores[0][-1])
print(f'  Score range: {score_range:.4f}')
if score_range < 0.05:
    print('  ⚠ Very narrow score range — routing is near-random')

## Complete Hierarchical RAG Pipeline

We now combine routing and generation into one end-to-end pipeline. The key difference from flat RAG is that the retrieval stage has explicit internal structure.

In [ ]:
class HierarchicalRAG:
    def answer(self, question):
        trace = three_level_retrieve(question)
        context = "\n\n".join([
            f"[{chunk['chunk_id']}] {chunk['text']}"
            for chunk in trace["chunks"]
        ])
        answer = generate(
            f"Use only the evidence below.\n\nEvidence:\n{context}\n\nQuestion: {question}\n\nAnswer:",
            max_new_tokens=170,
        )
        return {"trace": trace, "answer": answer}

pipeline = HierarchicalRAG()
response = pipeline.answer("How do transmission and flexibility incentives interact in a clean grid?")
print("TRACE:", response["trace"]["chapters"], response["trace"]["sections"])
print("\nANSWER:\n", response["answer"])

## Tree Visualization in Plain Text

It is often useful to print the hierarchy exactly as the retriever sees it. That makes debugging easier and helps learners see the relationship between the abstract index and the original corpus.

In [ ]:
for chapter, sections in corpus.items():
    print(chapter)
    for section, paragraphs in sections.items():
        print("  ->", section)
        for i, paragraph in enumerate(paragraphs):
            print("      -", f"p{i}", paragraph[:70] + "...")

## When Hierarchical Indices Help Most

Hierarchical indices are especially useful for long documents, books, manuals, and enterprise corpora where the first problem is often **routing** rather than ranking. They help less when the corpus is tiny or when each chunk already stands alone semantically.

Their cost is maintenance. You must keep summaries fresh, preserve metadata, and decide how many levels are worth the complexity. But when the corpus has real structure, hierarchy usually gives a better inductive bias than a single flat index.

In [ ]:
decision_rows = [
    ["Small flat FAQ", "Low", "Use flat retrieval"],
    ["Large handbook with sections", "High", "Use 2-level hierarchy"],
    ["Books / reports / many nested topics", "High", "Use 3-level hierarchy"],
    ["Rapidly changing micro-docs", "Medium", "Hierarchy may be too costly"],
]

for row in decision_rows:
    print(" | ".join(row))

## Performance Metrics Summary

We now measure precision@k, latency, and token usage across all three strategies
(flat, two-level, three-level) on a set of evaluation queries. This gives a concrete
comparison beyond individual examples.

In [ ]:
eval_queries = [
    {"q": "How can transmission reduce the need for storage?",
     "relevant_sections": {"Grid Planning::Transmission"}},
    {"q": "What incentives reward fast flexibility during evening peaks?",
     "relevant_sections": {"Policy and Markets::Performance Incentives"}},
    {"q": "How do batteries support frequency regulation?",
     "relevant_sections": {"Flexibility Resources::Batteries"}},
    {"q": "What problems arise on feeders with high rooftop solar?",
     "relevant_sections": {"Grid Planning::Distribution"}},
]

def precision_at_k(retrieved_sections, relevant_sections, k):
    """Fraction of top-k retrieved chunks whose section is relevant."""
    hits = sum(1 for s in retrieved_sections[:k] if s in relevant_sections)
    return hits / k if k > 0 else 0.0

results = []
for eq in eval_queries:
    q = eq['q']
    relevant = eq['relevant_sections']

    # Flat
    t0 = time.time()
    flat_res = flat_retrieve(q, k=4)
    flat_time = time.time() - t0
    flat_secs = [f"{r['section']}" for r in flat_res]
    # Approximate section matching for flat (section name only)
    flat_matched = [any(s.endswith(fs.split('::')[-1]) if '::' in fs else s == fs for s in relevant) for fs in flat_secs]
    flat_p = sum(flat_matched[:4]) / 4

    # Two-level
    t0 = time.time()
    _, two_res = hierarchical_retrieve(q, section_k=2, chunk_k=4)
    two_time = time.time() - t0
    two_secs = [r['section_id'] for r in two_res]
    two_p = precision_at_k(two_secs, relevant, 4)

    # Three-level
    t0 = time.time()
    three_res = three_level_retrieve(q, chapter_k=1, section_k=2, chunk_k=4)
    three_time = time.time() - t0
    three_secs = [f"{c['chapter']}::{c['section']}" for c in three_res['chunks']]
    three_p = precision_at_k(three_secs, relevant, 4)

    flat_tokens = sum(len(r['text'].split()) for r in flat_res)
    two_tokens = sum(len(r['text'].split()) for r in two_res)
    three_tokens = sum(len(c['text'].split()) for c in three_res['chunks'])

    results.append({
        'query': q[:50],
        'flat_p4': flat_p, 'two_p4': two_p, 'three_p4': three_p,
        'flat_ms': flat_time*1000, 'two_ms': two_time*1000, 'three_ms': three_time*1000,
        'flat_tok': flat_tokens, 'two_tok': two_tokens, 'three_tok': three_tokens,
    })

# Print summary table
print(f"{'Query':50} | {'Strategy':10} | {'P@4':>5} | {'Latency':>9} | {'Tokens':>6}")
print('-' * 95)
for r in results:
    for strategy, p, ms, tok in [
        ('Flat',    r['flat_p4'],  r['flat_ms'],  r['flat_tok']),
        ('2-Level', r['two_p4'],   r['two_ms'],   r['two_tok']),
        ('3-Level', r['three_p4'], r['three_ms'], r['three_tok']),
    ]:
        print(f"{r['query']:50} | {strategy:10} | {p:5.2f} | {ms:7.1f}ms | {tok:6}")
    print('-' * 95)

# Averages
n = len(results)
avg = lambda key: sum(r[key] for r in results) / n
print(f"\n{'AVERAGE':50} | {'Flat':10} | {avg('flat_p4'):5.2f} | {avg('flat_ms'):7.1f}ms | {avg('flat_tok'):6.0f}")
print(f"{'':50} | {'2-Level':10} | {avg('two_p4'):5.2f} | {avg('two_ms'):7.1f}ms | {avg('two_tok'):6.0f}")
print(f"{'':50} | {'3-Level':10} | {avg('three_p4'):5.2f} | {avg('three_ms'):7.1f}ms | {avg('three_tok'):6.0f}")

## Key Takeaways

Hierarchical retrieval is a structured answer to a structured problem. When documents have internal organization, the retriever should usually know that. Summary routing, section filtering, and local detail search are all simple ideas — but together they produce a much more disciplined retrieval pipeline.